# Assignment 4: SageMaker 프로젝트 생성
이 과제에서는 [SageMaker Projects](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects.html)를 사용하여 CI/CD 파이프라인을 생성합니다.

SageMaker Project는 Studio UX 또는 SageMaker API를 통해 환경에 프로비저닝할 수 있는 Cloud Formation 기반 템플릿입니다. 이러한 템플릿은 Service Catalog에서 관리됩니다. [제공되는 프로젝트 템플릿](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-sm.html)을 사용하거나 [사용자 정의 템플릿](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-custom.html)을 생성할 수 있습니다.

프로젝트는 재사용 가능하고 테스트되고 관리되는 구성 요소 또는 솔루션 청사진을 ML 환경에 제공하기 위한 권장 패턴입니다.

이 과제의 연습을 위한 코드 스니펫과 일반적인 지침은 [`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북을 참조하십시오.

## 패키지 임포트

In [ ]:
import boto3
import sagemaker 
from time import gmtime, strftime, sleep

In [ ]:
sm = boto3.client("sagemaker")
sc = boto3.client("servicecatalog")

sc_provider_name = "Amazon SageMaker"
sc_product_name = "MLOps template for model building and training"

## 연습 1: MLOps 프로젝트 생성
Python SDK `boto3`를 사용하여 프로그래밍 방식으로 프로젝트를 생성하려면 [`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북의 코드를 재사용할 수 있습니다.

또는 Studio UX를 통해 새 프로젝트를 프로비저닝할 수 있습니다.

In [ ]:
# 프로젝트를 생성한 후 프로젝트 세부 정보 가져오기
# project_name = <YOUR PROJECT NAME>
# sm.describe_project(ProjectName=project_name)

## 연습 2: 프로젝트 구성
프로비저닝된 프로젝트는 CodeCommit 리포지토리에 기본 구조와 시드 코드가 있는 템플릿입니다. Amazon S3 버킷의 이름, 특정 IAM 실행 역할 및 ML 워크플로와 같은 환경 및 사용 사례를 반영하도록 소스 코드와 일부 매개변수를 변경해야 합니다.

프로젝트 템플릿은 필수 구조가 아니며, 환경에 더 적합한 자체 사용자 정의 프로젝트 템플릿을 생성하기 위한 시작점을 제공할 뿐입니다.

프로비저닝된 프로젝트를 구성하려면:
1. 프로젝트 CodeCommit 리포지토리를 Studio EFS의 홈 디렉터리에 복제합니다
2. ML 파이프라인 구현 샘플 코드를 이전 과제에서 구현한 파이프라인 구성 코드로 교체합니다
3. 올바른 Python 모듈 이름을 참조하고 프로젝트 매개변수를 설정하도록 `codebuild-buildspec.yml` 파일을 수정합니다
4. `setup.py` 파일에서 잘못된 패키지 버전 요구 사항을 수정합니다

이 연습을 완료하기 위해 코드를 작성할 필요는 없으며, 구성 작업만 수행하면 됩니다.

### 프로젝트 시드 코드를 Studio 파일 시스템에 복제
복제 작업이 완료되면 홈 디렉터리의 프로젝트 폴더로 이동합니다.

### 처리 및 평가 스크립트를 파이프라인 폴더로 복사
[`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북의 지침을 따르십시오.

프로젝트의 코드 리포지토리 폴더 내부의 `pipelines` 폴더로 이동하여 `abalone` 폴더의 이름을 `fromideatoprod`로 변경합니다.

이전 두 과제에서 생성한 `preprocessing_assignment.py` 및 `evaluation_assignment.py` 스크립트를 프로젝트의 코드 리포지토리 폴더의 `pipelines/fromideatoprod` 폴더로 복사합니다.

### 파이프라인 구성 코드 교체
[`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북의 지침을 따르십시오. 파이프라인 구성 코드로 `pipeline.py` 파일을 생성합니다.

현재 폴더에서 이 `pipeline.py` 파일을 프로젝트의 코드 리포지토리 폴더의 `pipelines/fromideatoprod` 폴더로 복사합니다.

In [ ]:
%%writefile pipeline.py

# ../04-sagemaker-project.ipynb 노트북의 소스 코드를 재사용할 수 있습니다


### 빌드 사양 파일 수정
[`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북에 설명된 대로 새 빌드 사양 파일을 생성합니다.

입력 데이터셋의 S3 경로를 특정 S3 경로로 업데이트하는 것을 잊지 마십시오.

빌드 사양 파일을 프로젝트 폴더에 복사합니다.

In [ ]:
%%writefile codebuild-buildspec.yml

# Buildspec 파일

### `setup.py` 파일에서 잘못된 패키지 버전 수정
프로젝트 템플릿의 `setup.py` 파일에는 `sagemaker` 패키지에 대한 잘못된 버전 요구 사항이 포함되어 있습니다. 버전 번호를 제거하지 않으면 프로젝트 빌드가 실패합니다.

[`04-sagemaker-project.ipynb`](../04-sagemaker-project.ipynb) 노트북의 지침을 따르십시오.

## 연습 3: CI/CD 모델 빌드 파이프라인 실행

### CodePipeline 파이프라인
프로젝트 템플릿은 AWS 계정에 CodePipeline 파이프라인을 프로비저닝했습니다. CodePipeline [콘솔](https://console.aws.amazon.com/codesuite/codepipeline/pipelines)로 이동하여 모델 빌드 파이프라인과 해당 단계를 탐색합니다.

### 파이프라인을 시작하는 EventBridge 규칙
EventBridge [콘솔](https://console.aws.amazon.com/events/home?#/rules)의 규칙으로 이동하여 `sagemaker-<project-name>-<project-id>-build`라는 이름의 규칙을 찾습니다. 이 규칙은 프로젝트의 CodeCommit git 리포지토리의 각 변경 사항에서 CodePipeline 파이프라인을 시작합니다.

### 파이프라인 시작
코드 변경 사항을 리포지토리에 푸시하여 CI/CD 파이프라인을 시작합니다. 콘솔에서 파이프라인을 수동으로 시작할 수도 있습니다.

### 코드 커밋
변경한 코드를 커밋하고 푸시하려면 Studio의 Git 사이드 패널을 사용하거나 Studio 터미널에서 `git add`, `git commit` 및 `git push` 명령을 실행할 수 있습니다.

### 파이프라인 실행 추적
변경 사항을 리포지토리에 푸시한 후 파이프라인이 시작됩니다. CodePipeline [콘솔](https://console.aws.amazon.com/codesuite/codepipeline/pipelines)로 이동하여 실행을 확인합니다. 다양한 파이프라인 단계가 어떻게 함께 작동하여 ML 파이프라인을 빌드하고 실행하는지 탐색합니다.

## 연습 4: 모델 레지스트리 탐색
CI/CD 및 ML 파이프라인이 모두 성공적으로 실행되면 SageMaker 모델 레지스트리에 새 모델 버전이 등록됩니다.

**SageMaker resources** 드롭다운 목록에서 **Model registry**로 이동하여 모델 세부 정보를 확인합니다:

![](../img/model-package-group.png)

모델의 최신 버전을 더블 클릭합니다. 모델 세부 정보 화면의 모든 탭을 탐색합니다. 예를 들어, 이 모델 버전이 빌드된 전체 계보를 확인하고 버전 메타데이터를 탐색할 수 있습니다:

![](../img/model-version-details-annotated.png)

다음 과제에서는 모델 배포 파이프라인을 구현합니다.

## Assignment 5 계속하기
[assignment 5](05-assignment-deploy.ipynb) 노트북으로 이동합니다.